# OpsLM — inference benchmark session (Colab T4)

Produces the measured numbers for **Module 1/3/4** of the platform: vLLM vs Ollama,
FP16 vs AWQ vs GGUF Q4, across a concurrency sweep, plus the prefix-cache and
guided-decoding probes.

Runtime → **Change runtime type → T4 GPU**.

This notebook is a thin driver over `benchmarks/run_suite.py` in the repo (the script
is the source of truth). Each section writes one JSON file into `benchmarks/results/`;
the last section zips them for download so they can be committed.

**Budget ~90 min.** Sections 4–6 are the core; section 7 (AWQ) adds ~30 min and
section 9 (prefix-cache control) adds ~5 min but is what turns the prefix-cache
number into evidence rather than an anecdote.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

# T4 is Turing (sm_75): **no bfloat16**. Every server below is launched with
# --dtype float16 for that reason. On an Ampere+ box you can drop the flag.

## 2. Install vLLM

Use `uv pip install --torch-backend=auto`, not a bare `pip install vllm`. It reads the
**driver's** CUDA version and resolves vLLM and torch against the same CUDA major, which
avoids pinning versions that drift every few weeks.

The resolved version is captured below and stamped into every result file via `--notes`,
so reports stay reproducible.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version --format=csv,noheader

!pip -q install uv httpx
!uv pip install --system vllm --torch-backend=auto

### Put the CUDA runtime libraries on the loader path

**This step is required, and its absence is not obvious from the error it causes.**

A correct, self-consistent install still fails to import with:

```
ImportError: libcudart.so.13: cannot open shared object file
```

That message reads like a CUDA version mismatch. It is not. torch imports fine because
it loads CUDA libraries from its own bundled `torch/lib/`. vLLM's compiled extension is
a *separate* shared object that gets `dlopen`'d, and the dynamic loader has no idea where
the `nvidia-*` wheels put `libcudart.so.13`. Nothing is mismatched — the library is simply
unfindable.

Both fixes below are needed, for different consumers:

* `LD_LIBRARY_PATH` in `os.environ` — inherited by the **`vllm serve` subprocess**, which
  is what actually serves the model in sections 5 onward.
* `ctypes` preload with `RTLD_GLOBAL` — makes the symbols resolvable for **in-process**
  imports, which section 7 (AWQ) needs. Editing `LD_LIBRARY_PATH` alone cannot help there:
  the loader's search path is fixed when the process starts.

In [ ]:
import ctypes
import glob
import os
import subprocess
import sysconfig

SP = sysconfig.get_paths()["purelib"]

def find_cuda_libs():
    return glob.glob(f"{SP}/nvidia/**/libcudart.so.*", recursive=True)

hits = find_cuda_libs()
if not hits:
    print("CUDA runtime wheel absent — installing it")
    subprocess.run(["pip", "-q", "install", "nvidia-cuda-runtime-cu13"], check=True)
    hits = find_cuda_libs()
assert hits, "libcudart not found; send this output back before continuing"

libdirs = sorted({os.path.dirname(p) for p in hits})
libdirs += [d for d in (f"{SP}/torch/lib",) if os.path.isdir(d)]

# For subprocesses (vllm serve).
os.environ["LD_LIBRARY_PATH"] = ":".join(libdirs + [os.environ.get("LD_LIBRARY_PATH", "")])

# For in-process dlopen (the AWQ section). RTLD_GLOBAL exports the symbols so
# extensions loaded later can resolve against this copy.
for lib in hits:
    try:
        ctypes.CDLL(lib, mode=ctypes.RTLD_GLOBAL)
    except OSError as exc:
        print("preload skipped:", lib, exc)

print("lib dirs on path:", *libdirs, sep="\n  ")

### Verify via the CLI

Checked with `vllm --version` rather than `import vllm` on purpose: it exercises the same
subprocess path `serve()` uses below, so a pass here means the thing that actually matters
works.

In [ ]:
proc = subprocess.run(["vllm", "--version"], capture_output=True, text=True)
print(proc.stdout or proc.stderr)
assert proc.returncode == 0, "vllm CLI failed — check the CUDA library step above"
VLLM_VERSION = proc.stdout.strip().splitlines()[-1].strip()
print("VLLM_VERSION =", VLLM_VERSION)

## 3. Get the repo

Idempotent: safe to re-run after a session restart, when `/content/OPsVerse` may
already exist from an earlier attempt.

In [ ]:
%cd /content
![ -d OPsVerse ] || git clone --depth 1 https://github.com/dilipna/OPsVerse.git
%cd /content/OPsVerse
!git pull --ff-only 2>/dev/null || true
!mkdir -p benchmarks/results

MODEL_REPO = "dhf1234/OpsLM-v1"   # the merged 16-bit fine-tune
print("benchmarking:", MODEL_REPO)

## 4. Server control helpers

`serve()` launches a server in the background and blocks until `/v1/models` answers,
so a later measurement cell can never accidentally benchmark a half-loaded server —
that failure mode produces plausible-looking garbage.

In [ ]:
import subprocess, time, os, signal, urllib.request, json

SERVER = None

def stop():
    global SERVER
    if SERVER and SERVER.poll() is None:
        os.killpg(os.getpgid(SERVER.pid), signal.SIGTERM)
        SERVER.wait(timeout=120)
    SERVER = None
    time.sleep(5)   # let the GPU memory actually come back

def serve(args, port=8000, timeout_s=900):
    global SERVER
    stop()
    log = open(f"/content/server-{port}.log", "w")
    SERVER = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    url = f"http://localhost:{port}/v1/models"
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if SERVER.poll() is not None:
            !tail -40 /content/server-{port}.log
            raise RuntimeError(f"server exited with code {SERVER.returncode}")
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                print("ready:", json.load(r)["data"][0]["id"])
                return
        except Exception:
            time.sleep(5)
    raise TimeoutError(f"server not ready after {timeout_s}s — check /content/server-{port}.log")

## 5. vLLM @ FP16 — the baseline

`--enable-prefix-caching` is on so the prefix-cache probe has something to measure;
section 9 re-runs the same probe with it **off** as the control.

In [ ]:
serve([
    "vllm", "serve", MODEL_REPO,
    "--dtype", "float16",              # Turing has no bf16
    "--max-model-len", "4096",
    "--gpu-memory-utilization", "0.85",
    "--enable-prefix-caching",
    "--port", "8000",
])

In [ ]:
!python benchmarks/run_suite.py \
    --base-url http://localhost:8000/v1 \
    --model {MODEL_REPO} \
    --engine vllm --quant fp16 \
    --notes "vllm {VLLM_VERSION}, T4, prefix-caching on" \
    --concurrency 1,4,16 --requests 32 \
    --out benchmarks/results/vllm-opslm-fp16.json

## 6. Read the FP16 result before moving on

Sanity-check the sweep now: throughput should **rise** with concurrency while p95
latency rises sub-linearly. If throughput is flat, the server serialized the work and
something is wrong with the launch flags — fix it here rather than discovering it in
the report.

In [ ]:
import json
r = json.load(open("benchmarks/results/vllm-opslm-fp16.json"))
for lv in r["levels"]:
    print(f"c={lv['concurrency']:>3}  ttft_p50={lv['ttft_s']['p50']:.3f}s  "
          f"itl_p50={lv['itl_s']['p50']:.4f}s  thr={lv['throughput_tokens_s']:.1f} tok/s  "
          f"errors={lv['errors']}")
print("batching:", r["batching"])
print("prefix:", r["prefix_cache"])
print("structured:", r["structured_output"])

## 7. AWQ quantization (optional, ~30 min)

Calibrated on **`data/sft/train.jsonl` — the project's own in-domain training data**,
not a generic web corpus. AWQ picks which weight channels to protect based on observed
activations, so calibrating on the traffic the model will actually serve is the whole
point of doing it here rather than downloading someone else's quantization.

In [ ]:
stop()
!pip -q install autoawq

import json
calib = [json.loads(l) for l in open("data/sft/train.jsonl")][:256]
def to_text(row):
    msgs = row.get("messages") or []
    if msgs:
        return "\n".join(m.get("content", "") for m in msgs)
    return f"{row.get('instruction','')}\n{row.get('output','')}"
calib_text = [t for t in (to_text(r) for r in calib) if len(t) > 64]
print(len(calib_text), "calibration samples from the OpsLM SFT set")

In [ ]:
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer

QUANT_DIR = "/content/OpsLM-v1-awq"
tok = AutoTokenizer.from_pretrained(MODEL_REPO, trust_remote_code=True)
m = AutoAWQForCausalLM.from_pretrained(MODEL_REPO, safetensors=True, device_map="auto")
m.quantize(tok, quant_config={"zero_point": True, "q_group_size": 128,
                              "w_bit": 4, "version": "GEMM"},
           calib_data=calib_text)
m.save_quantized(QUANT_DIR); tok.save_pretrained(QUANT_DIR)
print("saved", QUANT_DIR)
del m; import torch, gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
serve([
    "vllm", "serve", QUANT_DIR,
    "--dtype", "float16",
    "--quantization", "awq",     # plain AWQ kernel; awq_marlin needs Ampere+
    "--max-model-len", "4096",
    "--gpu-memory-utilization", "0.85",
    "--enable-prefix-caching",
    "--served-model-name", "OpsLM-v1-awq",
    "--port", "8000",
])

In [ ]:
!python benchmarks/run_suite.py \
    --base-url http://localhost:8000/v1 \
    --model OpsLM-v1-awq \
    --engine vllm --quant awq \
    --notes "vllm {VLLM_VERSION}, T4, AWQ 4-bit GEMM calibrated on OpsLM SFT set" \
    --concurrency 1,4,16 --requests 32 \
    --out benchmarks/results/vllm-opslm-awq.json

## 8. Ollama @ GGUF Q4_K_M — the cross-engine comparison

Same model, same harness, same sweep, different engine. This is the row that shows
what a general-purpose runtime costs versus a dedicated serving engine — and it is
the engine actually running the always-on public demo endpoint, so its numbers are
the ones that describe what users experience.

In [ ]:
stop()
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
OLLAMA = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log","w"),
                          stderr=subprocess.STDOUT, start_new_session=True)
time.sleep(10)
!ollama pull hf.co/{MODEL_REPO}:Q4_K_M
!ollama list

In [ ]:
!python benchmarks/run_suite.py \
    --base-url http://localhost:11434/v1 \
    --model hf.co/{MODEL_REPO}:Q4_K_M \
    --engine ollama --quant q4_k_m \
    --notes "ollama, T4, GGUF Q4_K_M — same build serving the public demo" \
    --concurrency 1,4,16 --requests 32 \
    --out benchmarks/results/ollama-opslm-q4.json

## 9. Prefix-cache control run (~5 min)

Re-runs **only** the probes against vLLM with prefix caching **disabled**. Without
this control, a low warm-TTFT could be explained by warm-up effects; with it, the
difference between the two runs isolates the cache. Expect ≈0% reduction here.

In [ ]:
!pkill -f "ollama serve" || true
serve([
    "vllm", "serve", MODEL_REPO,
    "--dtype", "float16",
    "--max-model-len", "4096",
    "--gpu-memory-utilization", "0.85",
    "--no-enable-prefix-caching",
    "--port", "8000",
])

In [ ]:
!python benchmarks/run_suite.py \
    --base-url http://localhost:8000/v1 \
    --model {MODEL_REPO} \
    --engine vllm --quant fp16-noprefixcache \
    --notes "CONTROL: prefix caching disabled — isolates the cache effect" \
    --concurrency 1 --requests 8 \
    --out benchmarks/results/vllm-opslm-fp16-noprefix.json

## 10. Download the results

Commit these into `benchmarks/results/` in the repo. The raw JSON is committed
alongside the generated report so every chart stays reproducible.

In [ ]:
stop()
!ls -la benchmarks/results/
!cd benchmarks && zip -r /content/opslm-results.zip results/
from google.colab import files
files.download("/content/opslm-results.zip")

## 11. Back on the workstation

1. Unzip into `benchmarks/results/` and commit the JSON.
2. Get the **quality** axis — the frontier is meaningless without it. For each served
   config, run the Phase-4 eval and note the score:

   ```bash
   OPSVERSE_CHAT_MODEL=openai/OpsLM-v1 OPSVERSE_LLM_API_BASE=<endpoint> \
       uv run python -m opsverse_evals.rag_suite --n 20
   ```

3. Generate the report:

   ```bash
   python benchmarks/report.py --results benchmarks/results \
       --quality fp16=<score> --quality awq=<score> --quality q4_k_m=<score> \
       --out docs/reports/inference-benchmark-v1.md
   ```

Configs without a quality score are excluded from the frontier by design — a
speed-only frontier always recommends the smallest quantization, which is the exact
mistake the frontier exists to prevent.